# Free-field random observations

Observed-position rates are 2, 3, 5, 8, 10, 15 and 20%. Five seeds [1, 42, 123, 456, 789] vary the observation mask and sampling noise. All points, including 2%, are retained in the output table and plot.

Run cells in order. Sampling counts below are reverse-diffusion steps; checkpoint numbers identify training state. Data and checkpoints are installed under the repository root.


In [ ]:
from pathlib import Path
import os
import sys
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

cwd = Path.cwd().resolve()
ROOT = next((p for p in [cwd, *cwd.parents] if (p / 'models').is_dir() and (p / 'sampler').is_dir() and (p / 'trainer').is_dir()), None)
if ROOT is None:
    raise RuntimeError('Open this notebook from inside the MRSM repository.')
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
OUTPUT = ROOT / 'outputs' / 'free_field_random'
OUTPUT.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({'font.family': 'Arial', 'font.size': 10})

def format_current_figure():
    def tick(value, pos):
        return '0' if abs(value) < 1e-12 else f'{value:g}'
    for ax in plt.gcf().axes:
        if ax.get_xscale() == 'linear':
            ax.xaxis.set_major_formatter(FuncFormatter(tick))
        if ax.get_yscale() == 'linear':
            ax.yaxis.set_major_formatter(FuncFormatter(tick))

print('Repository:', ROOT)
print('Device:', device)


In [ ]:
import os
from sampler.utils import *
import importlib
import sampler.utils
from trainer.utils import restore_checkpoint
import torch
import torch.nn as nn
import json
from collections import namedtuple
from tqdm import tqdm
from models import get_model, get_ema
from sampler import VESDE
import numpy as np
from einops import rearrange
data_name = '2DSound'
version = 'v22'
opt_path = './results/' + data_name + '_' + version + '/config.json'
with open(opt_path, 'r', encoding='utf-8') as f:
    opt = json.load(f)
    opt['num_channels'] = opt['num_components'] + opt['num_conditions']
opt['results_path'] = str(Path(opt_path).parent)
opt['device'] = device
OPT_class = namedtuple('OPT_class', opt.keys())
opt = OPT_class(**opt)
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'


In [ ]:
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
net = get_model(opt)
net.to(device)
ema = get_ema(net.parameters(), decay=opt.ema_rate)
optimizer = torch.optim.Adam(net.parameters(), lr=opt.lr, betas=(0.9, 0.999), eps=1e-08, weight_decay=0)
state = dict(optimizer=optimizer, model=net, ema=ema, epoch=0, iteration=0, loss_train=[])
ckpt_path = opt.results_path + '/checkpoint_200.pth'
if not Path(ckpt_path).is_file():
    raise FileNotFoundError(ckpt_path)
state = restore_checkpoint(ckpt_path, state, device)
net = state['model']
ema = state['ema']
ema.copy_to(net.parameters())
print('Model loaded from epoch', state['epoch'])


In [ ]:
class DataScalar:

    def __init__(self, mean, std, device='cpu'):
        self.mean_np = mean
        self.std_np = std
        self.device = device
        self.mean_torch = torch.tensor(mean).float().to(device)
        self.std_torch = torch.tensor(std).float().to(device)
        if self.mean_torch.ndim == 5:
            self.mean_torch = self.mean_torch.permute(0, 1, 4, 2, 3)
            self.std_torch = self.std_torch.permute(0, 1, 4, 2, 3)

    def normalize(self, x_np):
        mean_t = self.mean_np.transpose(0, 1, 4, 2, 3)
        std_t = self.std_np.transpose(0, 1, 4, 2, 3)
        return (x_np - mean_t) / std_t

    def unnormalize(self, x):
        if x.device != self.mean_torch.device:
            self.mean_torch = self.mean_torch.to(x.device)
            self.std_torch = self.std_torch.to(x.device)
        return x * self.std_torch + self.mean_torch

def select_test_sample_by_freq(data_raw, frequency_hz=375):
    frequencies = [125, 135, 175, 225, 275, 325, 375, 415, 425]
    if frequency_hz not in frequencies:
        raise ValueError(f"Choose a frequency from {frequencies}")
    index = frequencies.index(frequency_hz)
    print(f"Test frequency: {frequency_hz} Hz")
    return data_raw[index:index + 1]


In [ ]:
data_test_path = 'data/anechoic_high_prec_dataset_last700_test_2.npy'
scalar_path = opt.results_path + '/scalar.npy'
data_raw = np.load(data_test_path)
scalar_data = np.load(scalar_path, allow_pickle=True)
(mean_val, std_val) = (scalar_data[0], scalar_data[1]) if isinstance(scalar_data, np.ndarray) else (scalar_data[0], scalar_data[1])
scaler = DataScalar(mean_val, std_val, device=device)
x_test_raw = select_test_sample_by_freq(data_raw, frequency_hz=375)
x_test = x_test_raw.transpose(0, 1, 4, 2, 3)
print(f'Sample Shape (Model Input): {x_test.shape}')
(_, z) = np.meshgrid(np.linspace(0, 2 * np.pi, opt.image_size, endpoint=False), np.linspace(0, 2 * np.pi, opt.image_size, endpoint=False), indexing='ij')


In [ ]:
m = 2
T = opt.num_frames
T_prime = 100
B_batch = int(np.ceil((T_prime - m) / (T - m)))
T_prime = B_batch * (T - m) + m
n_samples = 1
seed_samples = 1
rnd = np.random.RandomState(seed_samples)
t_start = 0
samples = x_test[:, t_start:t_start + T_prime]
print(f'Target Shape: {samples.shape}')
samples_scale_all = scaler.normalize(samples)
print(samples_scale_all.shape)


In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from tqdm import tqdm
import os
from contextlib import redirect_stdout, redirect_stderr
from scipy.stats import pearsonr

def cal_pearson_per_frame(true_data, pred_data):
    if true_data.shape != pred_data.shape:
        raise ValueError(f'Shapes must match: {true_data.shape} vs {pred_data.shape}')
    num_frames = true_data.shape[0]
    correlations = []
    for t in range(num_frames):
        true_flat = true_data[t].ravel()
        pred_flat = pred_data[t].ravel()
        (corr, _) = pearsonr(true_flat, pred_flat)
        if np.isnan(corr):
            correlations.append(0.0)
        else:
            correlations.append(corr)
    return np.array(correlations)

def unnorm_tensor(x_numpy):
    if x_numpy is None:
        return None
    t = torch.tensor(x_numpy).float().to(device)
    return scaler.unnormalize(t).detach().cpu().numpy()
observable_portions_pct = [2, 3, 5, 8, 10, 15, 20]
observable_portions = [p / 100.0 for p in observable_portions_pct]
scales_to_run = [1.0 - op for op in observable_portions]
random_seeds = [1, 42, 123, 456, 789]
plot_data = {'observable_pct': observable_portions_pct, 'sde_p_rmse_mean': [], 'sde_p_rmse_std': [], 'sde_vx_rmse_mean': [], 'sde_vx_rmse_std': [], 'sde_vy_rmse_mean': [], 'sde_vy_rmse_std': [], 'iss_p_rmse_mean': [], 'iss_p_rmse_std': [], 'iss_vx_rmse_mean': [], 'iss_vx_rmse_std': [], 'iss_vy_rmse_mean': [], 'iss_vy_rmse_std': [], 'pi_iss_p_rmse_mean': [], 'pi_iss_p_rmse_std': [], 'pi_iss_vx_rmse_mean': [], 'pi_iss_vx_rmse_std': [], 'pi_iss_vy_rmse_mean': [], 'pi_iss_vy_rmse_std': [], 'sde_p_corr_mean': [], 'sde_p_corr_std': [], 'sde_vx_corr_mean': [], 'sde_vx_corr_std': [], 'sde_vy_corr_mean': [], 'sde_vy_corr_std': [], 'iss_p_corr_mean': [], 'iss_p_corr_std': [], 'iss_vx_corr_mean': [], 'iss_vx_corr_std': [], 'iss_vy_corr_mean': [], 'iss_vy_corr_std': [], 'pi_iss_p_corr_mean': [], 'pi_iss_p_corr_std': [], 'pi_iss_vx_corr_mean': [], 'pi_iss_vx_corr_std': [], 'pi_iss_vy_corr_mean': [], 'pi_iss_vy_corr_std': []}
true_data_all = samples.squeeze(0)
true_data_p = true_data_all[:, 0:1, :, :]
true_data_vx = true_data_all[:, 1:2, :, :]
true_data_vy = true_data_all[:, 2:3, :, :]
print(f'True data shapes: p{true_data_p.shape}, vx{true_data_vx.shape}, vy{true_data_vy.shape}')
for (i, scale) in enumerate(tqdm(scales_to_run, desc='Processing Scales', dynamic_ncols=True, leave=True)):
    observable_pct = observable_portions_pct[i]
    base_keys = ['sde_p', 'sde_vx', 'sde_vy', 'iss_p', 'iss_vx', 'iss_vy', 'pi_iss_p', 'pi_iss_vx', 'pi_iss_vy']
    results_rmse = {k: [] for k in base_keys}
    results_corr = {k: [] for k in base_keys}
    for seed_sample in tqdm(random_seeds, desc=f'  Seeds (obs {observable_pct}%)', dynamic_ncols=True, leave=False):
        cof = 5e-08
        obs_type = 'random'
        alpha_case = 0.4
        beta = 0.4
        gamma = 0.5
        outer_loop = 100
        denoising_steps1 = outer_loop
        denoising_steps2 = outer_loop
        sde = VESDE(opt, sigma_min=opt.beta_min, sigma_max=opt.beta_max, N=outer_loop)
        sde1 = VESDE(opt, sigma_min=opt.beta_min, sigma_max=opt.beta_max, N=denoising_steps1)
        sde2 = VESDE(opt, sigma_min=opt.beta_min, sigma_max=opt.beta_max, N=denoising_steps2)
        net.eval()
        reap_fn = lambda x, num: torch.stack([x for _ in range(num)])
        setup_seed(seed_sample)
        with torch.no_grad():
            T_prime_y = T_prime
            mask = np.zeros([opt.num_components + opt.num_conditions, opt.image_size, opt.image_size])
            mask[:opt.num_components] = mask_gen([1, opt.image_size, opt.image_size], scale, seed=seed_sample)
            mask_torch = torch.from_numpy(mask).to(device)
            transform_gen = transform = lambda x: x[:, :T_prime_y] * mask[np.newaxis, np.newaxis] if isinstance(x, np.ndarray) else x[:, :T_prime_y] * mask_torch[None, None].to(x.device)
            sparsity = scale
            y = transform_gen(samples_scale_all)
            latent_mask = torch.ones([T, 1, 1, 1]).float().to(device)
            obs_mask = torch.zeros([T, 1, 1, 1]).float().to(device)
            frame_indices = torch.arange(T).to(device)
            net_fn = lambda x, t: net(x, x0=x, timesteps=t, latent_mask=reap_fn(latent_mask, len(x)), obs_mask=reap_fn(obs_mask, len(x)), frame_indices=reap_fn(frame_indices, len(x)))[0]
            with open(os.devnull, 'w') as fnull:
                with redirect_stdout(fnull), redirect_stderr(fnull):
                    x_inpainted = s3gm_sample_2d(opt, net_fn, sde, y, transform_gen, alpha=0.5 / np.sqrt(1 - sparsity), beta=beta, gamma=gamma, T_prime_y=T_prime_y, T_prime=T_prime, overlap=m, corrector=NoneCorrector, n_steps=0, snr=0.128, device=device, dtype='float', eps=1e-12, probability_flow=False, continuous=True)
                    setup_seed(seed_sample)
                    x_inpainted1 = s3gm_sample_2d_ISS_2dsound(opt, net_fn, sde1, y, transform_gen, alpha=0.5 / np.sqrt(1 - sparsity), beta=beta, gamma=gamma, T_prime_y=T_prime_y, T_prime=T_prime, overlap=m, corrector=NoneCorrector, n_steps=0, eta=0, reg_coef_pa=10 * T_prime, reg_coef_ar=100.0, denoising_steps=denoising_steps1, snr=0.128, device=device, dtype='float', eps=1e-12, probability_flow=False, continuous=True)
                    setup_seed(seed_sample)
                    x_inpainted2 = s3gm_sample_2d_ISS_2dsound_res(opt, net_fn, sde2, y, transform_gen, alpha=1.4 / np.sqrt(1 - sparsity), beta=beta, gamma=gamma, T_prime_y=T_prime_y, T_prime=T_prime, overlap=m, corrector=NoneCorrector, n_steps=0, eta=0, use_residual_objective=False, reg_coef_pa=10 * T_prime, reg_coef_ar=100.0, denoising_steps=denoising_steps2, snr=0.128, device=device, dtype='float', eps=1e-12, probability_flow=False, continuous=True, data_scalar=scaler.unnormalize, physics_start_step=100, reg_coef_wave=1 * cof, reg_coef_cont=100000000.0 * cof, reg_coef_euler=100000000.0 * cof, physics_c=343.0, physics_rho0=1.21, physics_dt=0.0001, physics_dx=1 / 21, physics_dy=1 / 21)
        x_pred = unnorm_tensor(x_inpainted)
        x_pred1 = unnorm_tensor(x_inpainted1)
        x_pred2 = unnorm_tensor(x_inpainted2)
        pred_data_sde = x_pred.squeeze(0)
        pred_data_iss = x_pred1.squeeze(0)
        pred_data_pi_iss = x_pred2.squeeze(0)
        corr_sde_p = np.mean(cal_pearson_per_frame(true_data_p, pred_data_sde[:, 0:1]))
        corr_sde_vx = np.mean(cal_pearson_per_frame(true_data_vx, pred_data_sde[:, 1:2]))
        corr_sde_vy = np.mean(cal_pearson_per_frame(true_data_vy, pred_data_sde[:, 2:3]))
        corr_iss_p = np.mean(cal_pearson_per_frame(true_data_p, pred_data_iss[:, 0:1]))
        corr_iss_vx = np.mean(cal_pearson_per_frame(true_data_vx, pred_data_iss[:, 1:2]))
        corr_iss_vy = np.mean(cal_pearson_per_frame(true_data_vy, pred_data_iss[:, 2:3]))
        corr_pi_iss_p = np.mean(cal_pearson_per_frame(true_data_p, pred_data_pi_iss[:, 0:1]))
        corr_pi_iss_vx = np.mean(cal_pearson_per_frame(true_data_vx, pred_data_pi_iss[:, 1:2]))
        corr_pi_iss_vy = np.mean(cal_pearson_per_frame(true_data_vy, pred_data_pi_iss[:, 2:3]))
        err_sde_p = np.mean(cal_rmse(true_data_p, pred_data_sde[:, 0:1], normalize=True))
        err_sde_vx = np.mean(cal_rmse(true_data_vx, pred_data_sde[:, 1:2], normalize=True))
        err_sde_vy = np.mean(cal_rmse(true_data_vy, pred_data_sde[:, 2:3], normalize=True))
        err_iss_p = np.mean(cal_rmse(true_data_p, pred_data_iss[:, 0:1], normalize=True))
        err_iss_vx = np.mean(cal_rmse(true_data_vx, pred_data_iss[:, 1:2], normalize=True))
        err_iss_vy = np.mean(cal_rmse(true_data_vy, pred_data_iss[:, 2:3], normalize=True))
        err_pi_iss_p = np.mean(cal_rmse(true_data_p, pred_data_pi_iss[:, 0:1], normalize=True))
        err_pi_iss_vx = np.mean(cal_rmse(true_data_vx, pred_data_pi_iss[:, 1:2], normalize=True))
        err_pi_iss_vy = np.mean(cal_rmse(true_data_vy, pred_data_pi_iss[:, 2:3], normalize=True))
        results_corr['sde_p'].append(corr_sde_p)
        results_corr['sde_vx'].append(corr_sde_vx)
        results_corr['sde_vy'].append(corr_sde_vy)
        results_corr['iss_p'].append(corr_iss_p)
        results_corr['iss_vx'].append(corr_iss_vx)
        results_corr['iss_vy'].append(corr_iss_vy)
        results_corr['pi_iss_p'].append(corr_pi_iss_p)
        results_corr['pi_iss_vx'].append(corr_pi_iss_vx)
        results_corr['pi_iss_vy'].append(corr_pi_iss_vy)
        results_rmse['sde_p'].append(err_sde_p)
        results_rmse['sde_vx'].append(err_sde_vx)
        results_rmse['sde_vy'].append(err_sde_vy)
        results_rmse['iss_p'].append(err_iss_p)
        results_rmse['iss_vx'].append(err_iss_vx)
        results_rmse['iss_vy'].append(err_iss_vy)
        results_rmse['pi_iss_p'].append(err_pi_iss_p)
        results_rmse['pi_iss_vx'].append(err_pi_iss_vx)
        results_rmse['pi_iss_vy'].append(err_pi_iss_vy)
        torch.cuda.empty_cache()
    for key in base_keys:
        plot_data[f'{key}_rmse_mean'].append(np.mean(results_rmse[key]))
        plot_data[f'{key}_rmse_std'].append(np.std(results_rmse[key]))
        plot_data[f'{key}_corr_mean'].append(np.mean(results_corr[key]))
        plot_data[f'{key}_corr_std'].append(np.std(results_corr[key]))
print("All processing done. Data stored in 'plot_data'.")
import pandas as pd
from datetime import datetime
save_dir = 'outputs/free_field_random/'
if not os.path.exists(save_dir):
    os.makedirs(save_dir, exist_ok=True)
df_results = pd.DataFrame(plot_data)
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
csv_name = f'experiment_results_{timestamp}.csv'
save_path = os.path.join(save_dir, csv_name)
try:
    df_results.to_csv(save_path, index=False)
    print(f'\n[SUCCESS] 实验数据已安全保存至: {save_path}')
except Exception as e:
    raise
print('\n预览保存的数据内容:')
print(df_results.head())


## Reconstruction error and correlation


In [ ]:
(fig, axes) = plt.subplots(1, 2, figsize=(11, 4))
for method in ['sde', 'iss', 'pi_iss']:
    for (ax, metric) in zip(axes, ['rmse', 'corr']):
        means = np.mean([plot_data[f'{method}_{ch}_{metric}_mean'] for ch in ['p', 'vx', 'vy']], axis=0)
        ax.plot(observable_portions_pct, means, marker='o', label=method.upper().replace('_', '-'))
axes[0].set(xlabel='Observed positions (%)', ylabel='Channel-averaged nRMSE', yscale='log')
axes[1].set(xlabel='Observed positions (%)', ylabel='Channel-averaged Pearson correlation')
for ax in axes:
    ax.legend()
format_current_figure()
fig.savefig(OUTPUT / 'random_observations.pdf', bbox_inches='tight')
format_current_figure()
plt.show()
